# 02 · Manipulación de los datos

Tomamos los datos limpios de `data/processed/` y los **reorganizamos**
para responder nuestras preguntas, sin modificar los archivos originales.

Cada base cuenta los viajes de forma distinta: Metro por **estación y día**, Metrobús por **línea y día**, Ecobici por **viaje** individual. Compararlas así sería como comparar lo que pasa en una estación contra lo que pasa en una línea. Por eso primero las **sumamos a una misma medida** (por ejemplo, pasajeros por línea por día) y después las juntamos.

El resultado son tablas listas para la parte de estadística y de visualización.

## 1. Cargar los datos limpios
Ubicamos la carpeta del proyecto sin escribir rutas fijas, para que el notebook funcione en cualquier computadora.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Buscamos la carpeta del proyecto desde donde está el notebook
raiz = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "data" / "processed" / "bitacora_limpieza.csv").is_file()), None)
if raiz is None:
    raise FileNotFoundError("No encuentro data/processed/. Ejecutar el notebook de limpieza.")

carpeta_datos = raiz / "data" / "processed"
print("Directorio raíz:", raiz.name)

Directorio raíz: Movilidad-Urbana-del-Sistema-de-Transporte-CDMX


### 1.1. Contenido por tabla
Revisamos el contenido y el periodo

In [2]:
# Leemos las tablas limpias y convertimos las fechas
metro = pd.read_csv(carpeta_datos / "metro_limpio.csv.gz", parse_dates=["fecha"])
metrobus = pd.read_csv(carpeta_datos / "metrobus_limpio.csv", parse_dates=["fecha"])
ingresos = pd.read_csv(carpeta_datos / "ingresos_limpio.csv.gz", parse_dates=["fecha"])
historico = pd.read_csv(carpeta_datos / "afluencia_historica_limpia.csv", parse_dates=["fecha"])
ecobici = pd.read_csv(carpeta_datos / "ecobici_2026_01_limpio.csv.gz", parse_dates=["fecha_retiro"],
                      dtype={"Ciclo_Estacion_Retiro": "string", "Ciclo_EstacionArribo": "string"})
bitacora_limpieza = pd.read_csv(carpeta_datos / "bitacora_limpieza.csv")

# Anotamos qué representa cada renglón y cuál es su columna de fecha
tablas = {"metro": (metro, "fecha", "estación × día × tipo de pago"),
          "metrobus": (metrobus, "fecha", "línea × día × tipo de pago"),
          "ingresos": (ingresos, "fecha", "línea × día × tipo de ingreso"),
          "historico": (historico, "fecha", "organismo/línea × día"),
          "ecobici": (ecobici, "fecha_retiro", "un viaje")}

# Armamos el inventario: filas, columnas y periodo de cada tabla
inventario = pd.DataFrame([
    {"tabla": nombre, "cada_renglon_es": unidad, "filas": len(tabla), "columnas": tabla.shape[1],
     "desde": tabla[columna_fecha].min().date(), "hasta": tabla[columna_fecha].max().date()}
    for nombre, (tabla, columna_fecha, unidad) in tablas.items()
])
display(inventario)

,tabla,cada_renglon_es,filas,columnas,desde,hasta
0,metro,estación × día × tipo de pago,1227915,10,2021-01-01,2026-09-30
1,metrobus,línea × día × tipo de pago,29386,8,2021-01-01,2026-09-30
2,ingresos,línea × día × tipo de ingreso,159936,6,2012-01-01,2023-07-31
3,historico,organismo/línea × día,18714,8,2020-03-01,2021-07-02
4,ecobici,un viaje,1525302,13,2025-08-02,2026-01-31


### 1.2. Verificar duplicados
Cada combinación de fecha, línea y tipo de pago debe aparecer una sola vez, de lo contrario se detendrá el cálculo.

In [3]:
# Definimos qué columnas identifican un solo renglón en cada tabla
llaves = {"metro": ["fecha", "linea", "estacion", "tipo_pago"],
          "metrobus": ["fecha", "linea", "tipo_pago"],
          "ingresos": ["fecha", "linea", "tipo_ingreso"]}

# Contamos duplicados; si aparece alguno, el notebook se detiene
for nombre, llave in llaves.items():
    duplicados = tablas[nombre][0].duplicated(llave).sum()
    print(nombre, "- duplicados:", duplicados)
    assert duplicados == 0

print("Pasos de limpieza previos:", bitacora_limpieza["paso"].unique().tolist())

metro - duplicados: 0
metrobus - duplicados: 0
ingresos - duplicados: 0
Pasos de limpieza previos: ['D1', 'D2', 'D3', 'D4', 'T']


## 2. Bitácora de decisiones
Igual que en la limpieza, registramos cada cambio sobre qué problemas existen y qué podemos hacer bajo algunas limitaciones

In [4]:
# Creamos la lista donde guardamos cada decisión
registro = []

# Agregamos una decisión a la bitácora
def registrar_decision(paso, tabla, variable, criterio, afectadas, accion, limitacion):
    registro.append({"paso": paso, "tabla": tabla, "variable": variable, "criterio": criterio,
                     "filas_afectadas": int(afectadas), "accion": accion, "limitacion": limitacion})

### 2.1. Un mes con el tipo de pago al revés
Al revisar qué proporción de pasajeros viaja gratis, febrero de 2025 sale muy distinto a los demás meses.

In [5]:
# Sumamos pasajeros por día y tipo de pago
diaria = metro.groupby(["fecha", "tipo_pago"])["afluencia"].sum().unstack()

# Calculamos el porcentaje de gratuidad y su mediana por mes
diaria["porcentaje_gratuidad"] = 100 * diaria["Gratuidad"] / diaria.sum(axis=1)
mensual = diaria.groupby(diaria.index.to_period("M"))["porcentaje_gratuidad"].median()
display(mensual.loc["2024-12":"2025-04"].round(1).to_frame("porcentaje_gratuidad_mediana"))

# Buscamos los días en que más de la mitad de los pasajeros aparece como gratuidad
dias_raros = diaria[diaria["porcentaje_gratuidad"] > 50]
print("Días con más de 50 % de gratuidad:", len(dias_raros))
print("Desde", dias_raros.index.min().date(), "hasta", dias_raros.index.max().date())

,porcentaje_gratuidad_mediana
fecha,
2024-12,12.7
2025-01,13.4
2025-02,87.2
2025-03,12.9
2025-04,13.1


Días con más de 50 % de gratuidad: 28
Desde 2025-02-01 hasta 2025-02-28


### 2.2. Corrección
Intercambiamos las etiquetas solo en ese mes, sobre una copia `metro_corregido`, y dejamos una columna que indica qué filas se corrigieron. El total de pasajeros no debería cambiar.

In [6]:
# Copiamos la tabla para no modificar la original
metro_corregido = metro.copy()

# Marcamos las filas de febrero 2025 que vamos a corregir
feb_2025 = metro_corregido["fecha"].dt.to_period("M").eq("2025-02")
metro_corregido["tipo_pago_corregido"] = feb_2025 & metro_corregido["tipo_pago"].isin(["Gratuidad", "Prepago"])

# Intercambiamos Gratuidad y Prepago solo en ese mes
metro_corregido.loc[feb_2025, "tipo_pago"] = metro_corregido.loc[feb_2025, "tipo_pago"].replace(
    {"Gratuidad": "Prepago", "Prepago": "Gratuidad"})

# Verificamos que el total no cambie y que no haya duplicados
assert metro_corregido["afluencia"].sum() == metro["afluencia"].sum()
assert metro_corregido.duplicated(["fecha", "linea", "estacion", "tipo_pago"]).sum() == 0

# Revisamos otra vez los días con más de 50 % de gratuidad
revision = metro_corregido.groupby(["fecha", "tipo_pago"])["afluencia"].sum().unstack()
revision["porcentaje_gratuidad"] = 100 * revision["Gratuidad"] / revision.sum(axis=1)
print("Días con más de 50 % de gratuidad después de corregir:", (revision["porcentaje_gratuidad"] > 50).sum())
print("Filas corregidas:", metro_corregido["tipo_pago_corregido"].sum())

# Registramos la decisión
registrar_decision("M1", "metro", "tipo_pago",
                   "feb-2025: Gratuidad ≈ 87 % y Prepago ≈ 13 % en los 28 días y las 12 líneas (lo normal es al revés)",
                   metro_corregido["tipo_pago_corregido"].sum(),
                   "intercambiar las etiquetas Gratuidad/Prepago en ese mes; bandera tipo_pago_corregido",
                   "supone un error de captura de la fuente; el total de pasajeros no cambia")

Días con más de 50 % de gratuidad después de corregir: 0
Filas corregidas: 10920


## 3. Transformación del sistema del Metro
Cada renglón de Metro es una estación en un día. Para comparar líneas (y después con Metrobús e ingresos), sumamos todas las estaciones de cada línea. Los tipos de pago pasan a ser columnas.

In [7]:
# Sumamos pasajeros por línea, día y tipo de pago; cada tipo de pago queda como columna
pasajeros = (metro_corregido.groupby(["fecha", "linea", "tipo_pago"])["afluencia"].sum()
                            .unstack("tipo_pago")
                            .reset_index())
pasajeros.columns.name = None
pasajeros = pasajeros.rename(columns={"Boleto": "boleto", "Prepago": "prepago", "Gratuidad": "gratuidad"})

# Contamos cuántas estaciones tiene cada línea y cuántas abrieron ese día
estaciones = (metro_corregido.drop_duplicates(["fecha", "linea", "estacion"])
                             .groupby(["fecha", "linea"])
                             .agg(estaciones=("estacion", "size"),
                                  estaciones_abiertas=("en_servicio", "sum"))
                             .reset_index())

# Unimos ambas tablas y calculamos el total de pasajeros
metro_linea_dia = pasajeros.merge(estaciones, on=["fecha", "linea"], how="left", validate="one_to_one")
metro_linea_dia["total"] = metro_linea_dia[["boleto", "prepago", "gratuidad"]].sum(axis=1)

### 3.1. Conservación de pasajeros
El total de pasajeros antes y después tiene que ser idéntico.

In [8]:
# Verificamos que no haya duplicados ni se pierdan pasajeros
assert metro_linea_dia[["fecha", "linea"]].duplicated().sum() == 0
assert metro_linea_dia["total"].sum() == metro_corregido["afluencia"].sum()
assert len(metro_linea_dia) == metro_corregido["fecha"].nunique() * metro_corregido["linea"].nunique()

print("Filas antes:", len(metro_corregido), "- después:", len(metro_linea_dia))
print("Pasajeros antes:", metro_corregido["afluencia"].sum(), "- después:", metro_linea_dia["total"].sum())

# Registramos la decisión
registrar_decision("M2", "metro", "afluencia", "estación × día × tipo de pago → línea × día",
                   len(metro_corregido), "sumar estaciones; tipos de pago como columnas",
                   "se pierde el detalle por estación (sigue disponible en metro_corregido)")

Filas antes: 1227915 - después: 25188
Pasajeros antes: 6298847633 - después: 6298847633


### 3.2. Variables nuevas
- `linea_abierta`: la línea tuvo pasajeros ese día (si suma 0, estuvo cerrada).
- `cierre_parcial`: la línea operó, pero con alguna estación cerrada.
- `porcentaje_boleto`, `porcentaje_prepago`, `porcentaje_gratuidad`: qué porcentaje de los pasajeros pagó de cada forma (de 0 a 100). Si la línea estuvo cerrada, se deja vacío.
- `mes`, `dia_semana`, `tipo_dia`: para comparar por calendario.

In [9]:
dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

# Marcamos si la línea abrió y si tuvo alguna estación cerrada
metro_linea_dia["linea_abierta"] = metro_linea_dia["total"].gt(0)
metro_linea_dia["cierre_parcial"] = (metro_linea_dia["linea_abierta"]
                                     & metro_linea_dia["estaciones_abiertas"].lt(metro_linea_dia["estaciones"]))

# Calculamos el porcentaje de cada tipo de pago (vacío si la línea estuvo cerrada)
for tipo in ["boleto", "prepago", "gratuidad"]:
    metro_linea_dia[f"porcentaje_{tipo}"] = np.where(metro_linea_dia["total"].gt(0),
                                                     100 * metro_linea_dia[tipo] / metro_linea_dia["total"], np.nan)

# Agregamos mes, día de la semana y tipo de día
metro_linea_dia["mes"] = metro_linea_dia["fecha"].dt.to_period("M").astype("string")
metro_linea_dia["dia_semana"] = pd.Categorical(metro_linea_dia["fecha"].dt.dayofweek.map(dict(enumerate(dias))),
                                               categories=dias, ordered=True)
metro_linea_dia["tipo_dia"] = np.where(metro_linea_dia["fecha"].dt.dayofweek >= 5, "Fin de semana", "Laboral")
display(metro_linea_dia.head())

,fecha,linea,boleto,gratuidad,prepago,estaciones,estaciones_abiertas,total,linea_abierta,cierre_parcial,porcentaje_boleto,porcentaje_prepago,porcentaje_gratuidad,mes,dia_semana,tipo_dia
0,2021-01-01,Línea 1,69381,13533,46664,20,20,129578,True,False,53.543811,36.012286,10.443903,2021-01,Viernes,Laboral
1,2021-01-01,Línea 12,0,15390,39226,20,20,54616,True,False,0.000000,71.821444,28.178556,2021-01,Viernes,Laboral
2,2021-01-01,Línea 2,39314,13143,39425,24,24,91882,True,False,42.787488,42.908295,14.304216,2021-01,Viernes,Laboral
3,2021-01-01,Línea 3,41833,11587,43300,21,21,96720,True,False,43.251654,44.768404,11.979942,2021-01,Viernes,Laboral
4,2021-01-01,Línea 4,7168,2919,4199,10,10,14286,True,False,50.174997,29.392412,20.432591,2021-01,Viernes,Laboral


### 3.3. Resumen por línea
La mediana de pasajeros se calcula solo con los días en que la línea estuvo abierta,
para no mezclar cierres con días normales (decisión D2 de la limpieza).

In [10]:
# Resumimos cada línea: días, cierres y gratuidad
resumen_lineas = (metro_linea_dia.groupby("linea")
                                 .agg(dias=("fecha", "size"),
                                      dias_cerrada=("linea_abierta", lambda s: int((~s).sum())),
                                      dias_cierre_parcial=("cierre_parcial", "sum"),
                                      mediana_porcentaje_gratuidad=("porcentaje_gratuidad", "median")))

# Calculamos la mediana de pasajeros usando solo los días en que la línea abrió
dias_abiertos = metro_linea_dia.loc[metro_linea_dia["linea_abierta"]]
resumen_lineas["mediana_pasajeros_abierta"] = dias_abiertos.groupby("linea")["total"].median()
display(resumen_lineas.sort_values("mediana_pasajeros_abierta", ascending=False).round(1))

,dias,dias_cerrada,dias_cierre_parcial,mediana_porcentaje_gratuidad,mediana_pasajeros_abierta
linea,,,,,
Línea 2,2099,30,293,11.9,556664.0
Línea 3,2099,23,39,12.0,472032.0
Línea B,2099,0,2,12.8,346372.0
Línea 8,2099,0,1,13.1,336828.0
Línea 12,2099,621,382,11.7,255976.5
Línea 1,2099,16,1293,17.3,250024.0
Línea 9,2099,0,279,9.5,242935.0
Línea 7,2099,0,141,10.1,216958.0
Línea A,2099,0,0,11.3,206580.0


## 4. Metrobús por línea y día
Metrobús ya viene por línea y día, así que solo pasamos los tipos de pago a columnas.
Metrobús no registra Boleto a diferencia del metro.

In [11]:
# Pasamos los tipos de pago a columnas
metrobus_linea_dia = (metrobus.pivot_table(index=["fecha", "linea"], columns="tipo_pago",
                                           values="afluencia", aggfunc="sum")
                              .reset_index())
metrobus_linea_dia.columns.name = None
metrobus_linea_dia = metrobus_linea_dia.rename(columns={"Prepago": "prepago", "Gratuidad": "gratuidad"})

# Calculamos el total y el porcentaje de gratuidad
metrobus_linea_dia["total"] = metrobus_linea_dia["prepago"] + metrobus_linea_dia["gratuidad"]
metrobus_linea_dia["porcentaje_gratuidad"] = 100 * metrobus_linea_dia["gratuidad"] / metrobus_linea_dia["total"]

# Verificamos que no haya duplicados ni se pierdan pasajeros
assert metrobus_linea_dia[["fecha", "linea"]].duplicated().sum() == 0
assert metrobus_linea_dia["total"].sum() == metrobus["afluencia"].sum()
print("Filas antes:", len(metrobus), "- después:", len(metrobus_linea_dia))
print("Celdas vacías:", metrobus_linea_dia.isna().sum().sum())

Filas antes: 29386 - después: 14693
Celdas vacías: 0


### 4.1. Metro y Metrobús en una misma tabla
Sumamos todas las líneas de cada sistema por día. Así cada renglón es **un sistema en un día**
y ambos quedan en la misma medida para compararlos.

In [12]:
# Sumamos todas las líneas de cada sistema por día y juntamos ambos sistemas
sistemas_dia = pd.concat([
    metro_linea_dia.groupby("fecha")[["prepago", "gratuidad", "boleto"]].sum().assign(sistema="Metro"),
    metrobus_linea_dia.groupby("fecha")[["prepago", "gratuidad"]].sum().assign(boleto=0, sistema="Metrobús"),
]).reset_index()

# Calculamos el total, el porcentaje de gratuidad y el calendario
sistemas_dia["total"] = sistemas_dia[["prepago", "gratuidad", "boleto"]].sum(axis=1)
sistemas_dia["porcentaje_gratuidad"] = 100 * sistemas_dia["gratuidad"] / sistemas_dia["total"]
sistemas_dia["mes"] = sistemas_dia["fecha"].dt.to_period("M").astype("string")
sistemas_dia["tipo_dia"] = np.where(sistemas_dia["fecha"].dt.dayofweek >= 5, "Fin de semana", "Laboral")

# Verificamos que cada sistema aparezca una sola vez por día
assert sistemas_dia[["fecha", "sistema"]].duplicated().sum() == 0

# Comparamos días laborales contra fines de semana
display(sistemas_dia.groupby(["sistema", "tipo_dia"])
                    .agg(dias=("fecha", "size"), mediana_pasajeros=("total", "median"),
                         mediana_porcentaje_gratuidad=("porcentaje_gratuidad", "median")).round(1))

# Registramos la decisión
registrar_decision("M3", "metro + metrobus", "afluencia", "unir ambos sistemas a sistema × día",
                   len(sistemas_dia), "sumar líneas; boleto = 0 en Metrobús porque no existe",
                   "Metrobús no distingue boleto; comparar tipo de pago solo con prepago y gratuidad")

dias  mediana_pasajeros  mediana_porcentaje_gratuidad
sistema  tipo_dia                                                            
Metro    Fin de semana   600          2076753.5                          11.9
         Laboral        1499          3455312.0                          12.4
Metrobús Fin de semana   600           833927.5                           6.5
         Laboral        1499          1542818.0                           6.2

## 5. Ingresos y pasajeros del Metro (2021–2023)
Los ingresos solo existen hasta julio de 2023 y los pasajeros desde enero de 2021, así que trabajamos
en el periodo que comparten. Usamos **meses** y no días: una recarga de tarjeta se compra un día
y se gasta en muchos viajes después, por lo que día con día los ingresos y los viajes no coinciden.

In [13]:
fecha_inicio, fecha_fin = "2021-01-01", "2023-07-31"

# Filtramos los ingresos al periodo que comparten ambas bases
ingresos_periodo = ingresos.loc[ingresos["fecha"].between(fecha_inicio, fecha_fin)].copy()
ingresos_periodo["mes"] = ingresos_periodo["fecha"].dt.to_period("M").astype("string")

# Sumamos los ingresos por línea y mes; cada tipo de ingreso queda como columna
ingresos_mes = (ingresos_periodo.pivot_table(index=["linea", "mes"], columns="tipo_ingreso",
                                             values="ingreso", aggfunc="sum", fill_value=0)
                                .reset_index())
ingresos_mes.columns.name = None
ingresos_mes = ingresos_mes.rename(columns={"Boletos": "ingreso_boletos", "Recargas": "ingreso_recargas",
                                            "Tarjetas": "ingreso_tarjetas", "QR/Validación": "ingreso_qr"})
ingresos_mes["ingreso_total"] = ingresos_mes[["ingreso_boletos", "ingreso_recargas",
                                              "ingreso_tarjetas", "ingreso_qr"]].sum(axis=1)

# Sumamos los pasajeros por línea y mes en el mismo periodo
pasajeros_mes = (metro_linea_dia.loc[metro_linea_dia["fecha"].between(fecha_inicio, fecha_fin)]
                                .groupby(["linea", "mes"])
                                .agg(pasajeros=("total", "sum"), boleto=("boleto", "sum"),
                                     prepago=("prepago", "sum"), gratuidad=("gratuidad", "sum"),
                                     dias_abierta=("linea_abierta", "sum"))
                                .reset_index())
pasajeros_mes["pagantes"] = pasajeros_mes["boleto"] + pasajeros_mes["prepago"]

### 5.1. Unión segura
Ambas tablas tienen ahora la misma medida (línea × mes), así que cada renglón debe encontrar
exactamente una pareja. `indicator` nos muestra si algún renglón se quedó sin pareja.

In [14]:
# Unimos ambas tablas y revisamos si algún renglón quedó sin pareja
ingresos_pasajeros = pasajeros_mes.merge(ingresos_mes, on=["linea", "mes"], how="outer",
                                         validate="one_to_one", indicator=True)
display(ingresos_pasajeros["_merge"].value_counts().to_frame("filas"))
assert ingresos_pasajeros["_merge"].eq("both").all()

# Quitamos la columna de revisión
ingresos_pasajeros = ingresos_pasajeros.drop(columns="_merge")

,filas
_merge,
both,372
left_only,0
right_only,0


### 5.2. Ingreso por pasajero que paga
Dividimos el ingreso entre los pasajeros que pagan (Boleto + Prepago; la gratuidad no genera ingreso).
Si las bases cuadran, el resultado debe estar cerca de la tarifa del Metro: **$5**.
Los meses en que la Línea 12 estuvo cerrada quedan vacíos.

In [15]:
# Marcamos los meses en que la línea estuvo cerrada
ingresos_pasajeros["linea_cerrada"] = ingresos_pasajeros["dias_abierta"].eq(0)

# Calculamos el ingreso por pasajero que paga (vacío si no hubo pagantes)
ingresos_pasajeros["ingreso_por_pagante"] = np.where(ingresos_pasajeros["pagantes"].gt(0),
                                                     ingresos_pasajeros["ingreso_total"] / ingresos_pasajeros["pagantes"],
                                                     np.nan)
print("Meses-línea cerrados:", ingresos_pasajeros["linea_cerrada"].sum())
display(ingresos_pasajeros.groupby("linea")["ingreso_por_pagante"]
                          .describe()[["count", "min", "50%", "max"]].round(2))

# Sumamos todo el Metro por mes (tabla principal de la pregunta 3)
metro_ingresos_mes = (ingresos_pasajeros.groupby("mes")[["pasajeros", "pagantes", "gratuidad",
                                                         "ingreso_boletos", "ingreso_recargas", "ingreso_tarjetas",
                                                         "ingreso_qr", "ingreso_total"]]
                                        .sum().reset_index())
metro_ingresos_mes["ingreso_por_pagante"] = metro_ingresos_mes["ingreso_total"] / metro_ingresos_mes["pagantes"]
display(metro_ingresos_mes.head())

# Registramos la decisión
registrar_decision("M4", "ingresos + metro", "ingreso", "unir ingresos y pasajeros por línea × mes (ene-2021 a jul-2023)",
                   len(ingresos_pasajeros), "agregar por mes; ingreso_por_pagante = ingreso / (boleto + prepago)",
                   "la recarga se vende en una línea y puede usarse en otra: comparar por línea con cautela")

Meses-línea cerrados: 19


,count,min,50%,max
linea,,,,
Línea 1,31.0,4.56,5.22,5.61
Línea 12,12.0,5.44,5.80,6.67
Línea 2,31.0,4.97,5.40,5.74
Línea 3,31.0,4.84,5.31,5.73
Línea 4,31.0,3.40,5.16,5.66
Línea 5,31.0,4.54,4.82,5.30
Línea 6,31.0,3.60,4.29,4.49
Línea 7,31.0,5.30,5.86,6.13
Línea 8,31.0,4.52,4.98,5.28


,mes,pasajeros,pagantes,gratuidad,ingreso_boletos,ingreso_recargas,ingreso_tarjetas,ingreso_qr,ingreso_total,ingreso_por_pagante
0,2021-01,41033642,36287874,4745768,47454875.0,1.440367e+08,1484355.0,0.0,1.929760e+08,5.317918
1,2021-02,50436321,46141886,4294435,64846325.0,1.786339e+08,1705425.0,0.0,2.451857e+08,5.313733
2,2021-03,65449779,60433657,5016122,85644965.0,2.264336e+08,2241675.0,0.0,3.143203e+08,5.201080
3,2021-04,62577359,57661064,4916295,82950270.0,2.087748e+08,2150445.0,0.0,2.938756e+08,5.096603
4,2021-05,62689975,56985969,5704006,94883350.0,1.894670e+08,1648830.0,0.0,2.859992e+08,5.018766


## 6. Ecobici: de viaje individual a día y hora
Cada renglón de Ecobici es un viaje. Primero quitamos los viajes marcados en la limpieza
(más de 24 h o iniciados antes de enero) y después contamos cuántos viajes hubo por
día, hora, género y rango de edad.

In [16]:
# Filtramos los viajes válidos según las banderas de la limpieza
validos = ~ecobici["duracion_imposible"] & ~ecobici["fuera_de_periodo"]
ecobici_validos = ecobici.loc[validos].copy()
print("Viajes válidos:", len(ecobici_validos), "de", len(ecobici))
print("Viajes excluidos:", (~validos).sum())

# Separamos el día y conservamos a quienes no tienen edad registrada
ecobici_validos["fecha"] = ecobici_validos["fecha_retiro"].dt.normalize()
ecobici_validos["rango_edad"] = ecobici_validos["rango_edad"].fillna("Sin edad")

# Contamos viajes y duración mediana por día, hora, género y edad
ecobici_hora = (ecobici_validos.groupby(["fecha", "hora_retiro", "genero", "rango_edad"])
                               .agg(viajes=("duracion_min", "size"),
                                    duracion_mediana=("duracion_min", "median"))
                               .reset_index())

# Agregamos día de la semana y tipo de día
ecobici_hora["dia_semana"] = pd.Categorical(ecobici_hora["fecha"].dt.dayofweek.map(dict(enumerate(dias))),
                                            categories=dias, ordered=True)
ecobici_hora["tipo_dia"] = np.where(ecobici_hora["fecha"].dt.dayofweek >= 5, "Fin de semana", "Laboral")

# Verificamos que no se pierdan viajes ni haya duplicados
assert ecobici_hora["viajes"].sum() == len(ecobici_validos)
assert ecobici_hora.duplicated(["fecha", "hora_retiro", "genero", "rango_edad"]).sum() == 0
print("Renglones resultantes:", len(ecobici_hora))

# Registramos la decisión
registrar_decision("M5", "ecobici", "viajes", "viaje → día × hora × género × rango de edad",
                   len(ecobici_validos), "excluir banderas D4 y contar viajes; mediana de duración por grupo",
                   "se pierde el detalle por estación; enero 2026 no permite ver estacionalidad")

Viajes válidos: 1525178 de 1525302
Viajes excluidos: 124


Renglones resultantes: 12995


### 6.1. Viajes promedio por hora
Dividimos entre el número de días de cada tipo (22 laborales y 9 de fin de semana),
porque si no, los días laborales parecerían más usados solo por ser más.

In [17]:
# Sumamos los viajes por hora y tipo de día
viajes_por_hora = ecobici_hora.pivot_table(index="hora_retiro", columns="tipo_dia", values="viajes", aggfunc="sum")

# Dividimos entre el número de días de cada tipo para obtener el promedio diario
viajes_por_hora = viajes_por_hora / ecobici_hora.groupby("tipo_dia")["fecha"].nunique()
display(viajes_por_hora.round(0).astype(int).T)

hora_retiro,0,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
tipo_dia,,,,,,,,,,,,,,,,,,,,
Fin de semana,262,120,362,777,1369,1984,2526,2913,3087,3104,2862,2644,2332,2138,1943,1610,1337,1088,825,539
Laboral,153,461,1552,3234,4349,3180,2373,2248,2408,3027,3777,3918,3829,4735,5289,4041,2813,2052,1318,733


### 6.2. Los tres sistemas en enero 2026
Enero 2026 es el único mes en que tenemos Metro, Metrobús y Ecobici al mismo tiempo.
Juntamos sus viajes por día en una sola tabla.

In [18]:
# Contamos los viajes de Ecobici por día
ecobici_dia = ecobici_validos.groupby("fecha").size().rename("total").reset_index().assign(sistema="Ecobici")

# Juntamos los tres sistemas en enero 2026
enero_2026 = pd.concat([sistemas_dia.loc[sistemas_dia["mes"].eq("2026-01"), ["fecha", "sistema", "total"]],
                        ecobici_dia], ignore_index=True)
enero_2026["tipo_dia"] = np.where(enero_2026["fecha"].dt.dayofweek >= 5, "Fin de semana", "Laboral")
assert enero_2026[["fecha", "sistema"]].duplicated().sum() == 0

# Comparamos la mediana de fin de semana contra la de día laboral
comparacion = enero_2026.pivot_table(index="sistema", columns="tipo_dia", values="total", aggfunc="median")
comparacion["finde_entre_laboral"] = comparacion["Fin de semana"] / comparacion["Laboral"]
display(comparacion.round(2))

tipo_dia,Fin de semana,Laboral,finde_entre_laboral
sistema,,,
Ecobici,34438.0,59092.5,0.58
Metro,2815078.0,3731433.0,0.75
Metrobús,996497.0,1546643.5,0.64


## 7. Guardar resultados
Guardamos las tablas en `data/processed/` con nombres que empiezan con `tabla_`, para distinguirlas
de los datos limpios. La bitácora va en un archivo aparte, así no se modifica la de la limpieza.

In [19]:
# Definimos qué tabla va en cada archivo y para qué pregunta sirve
salidas = {
    "tabla_metro_linea_dia.csv":              (metro_linea_dia,    "línea del Metro × día",              "P1"),
    "tabla_metrobus_linea_dia.csv":           (metrobus_linea_dia, "línea del Metrobús × día",           "P1"),
    "tabla_sistemas_dia.csv":                 (sistemas_dia,       "sistema (Metro/Metrobús) × día",     "P1"),
    "tabla_ingresos_pasajeros_linea_mes.csv": (ingresos_pasajeros, "línea del Metro × mes (2021–2023)",  "P3"),
    "tabla_metro_ingresos_mes.csv":           (metro_ingresos_mes, "todo el Metro × mes (2021–2023)",    "P3"),
    "tabla_ecobici_dia_hora.csv":             (ecobici_hora,       "día × hora × género × edad",         "P2"),
    "tabla_enero_2026_sistemas.csv":          (enero_2026,         "sistema (los tres) × día, ene-2026", "P2"),
}

# Guardamos cada tabla y anotamos su tamaño
resumen_salidas = []
for archivo, (tabla, unidad, pregunta) in salidas.items():
    tabla.to_csv(carpeta_datos / archivo, index=False)
    resumen_salidas.append({"archivo": archivo, "cada_renglon_es": unidad, "pregunta": pregunta,
                            "filas": len(tabla), "columnas": tabla.shape[1]})
display(pd.DataFrame(resumen_salidas))

,archivo,cada_renglon_es,pregunta,filas,columnas
0,tabla_metro_linea_dia.csv,línea del Metro × día,P1,25188,16
1,tabla_metrobus_linea_dia.csv,línea del Metrobús × día,P1,14693,6
2,tabla_sistemas_dia.csv,sistema (Metro/Metrobús) × día,P1,4198,9
3,tabla_ingresos_pasajeros_linea_mes.csv,línea del Metro × mes (2021–2023),P3,372,15
4,tabla_metro_ingresos_mes.csv,todo el Metro × mes (2021–2023),P3,31,10
5,tabla_ecobici_dia_hora.csv,día × hora × género × edad,P2,12995,8
6,tabla_enero_2026_sistemas.csv,"sistema (los tres) × día, ene-2026",P2,93,4


### 7.1. Bitácora completa
Unimos las decisiones de limpieza (D1–D4, T) con las de manipulación (M1–M5) para tener todo el recorrido.

In [20]:
# Guardamos la bitácora de manipulación en un archivo aparte
bitacora_manipulacion = pd.DataFrame(registro)
bitacora_manipulacion.to_csv(carpeta_datos / "bitacora_manipulacion.csv", index=False)

# Mostramos la bitácora completa: limpieza y manipulación
display(pd.concat([bitacora_limpieza, bitacora_manipulacion], ignore_index=True)[["paso", "tabla", "variable", "accion"]])

# Anotamos de dónde salen las tablas
(carpeta_datos / "origen_manipulacion.txt").write_text(
    "Origen: tablas limpias de data/processed/ (notebook Proyecto_Movilidad_CDMX)\n"
    "Transformación: notebooks/02_Manipulacion_Datos.ipynb\n"
    "Los archivos de limpieza no se modificaron.\n", encoding="utf8")
print("Tablas guardadas en data/processed/")

,paso,tabla,variable,accion
0,D1,metro,linea,estandarizar a 'Línea X'
1,D1,metro,estacion,reparar codificación y unificar a la escritura...
2,D1,metrobus,linea,estandarizar a 'Línea X'
3,D1,ingresos,tipo_ingreso,unificar en 'QR/Validación'
4,D2,metro,afluencia,conservar filas y crear bandera en_servicio; p...
5,D3,ingresos,todas,eliminar copia
6,D3,ingresos,ingreso,conservar el valor distinto de 0
7,D3,ingresos,ingreso,no imputar; columna motivo_faltante; excluir c...
8,D4,ecobici,Genero_Usuario,categoría explícita 'No especificado' (no impu...
9,D4,ecobici,Edad_Usuario,no imputar; quedan fuera solo de los análisis ...


Tablas guardadas en data/processed/


### Resumen

¿Qué archivo usar para cada pregunta?

| Pregunta | Archivo | Cada renglón es |
|:--|:--|:--|
| P1 · Tipo de pago | tabla_metro_linea_dia.csv | una línea del Metro en un día |
| P1 · Tipo de pago | tabla_metrobus_linea_dia.csv | una línea del Metrobús en un día |
| P1 · Tipo de pago | tabla_sistemas_dia.csv | un sistema (Metro o Metrobús) en un día |
| P2 · Ecobici | tabla_ecobici_dia_hora.csv | un grupo de género y edad en una hora de un día |
| P2 · Ecobici | tabla_enero_2026_sistemas.csv | un sistema (los tres) en un día de enero 2026 |
| P3 · Ingresos | tabla_metro_ingresos_mes.csv | todo el Metro en un mes (2021–2023) |
| P3 · Ingresos | tabla_ingresos_pasajeros_linea_mes.csv | una línea del Metro en un mes (2021–2023) |

Cada tabla conserva el total de pasajeros o viajes de su fuente (lo verificamos con `assert`).